# LeetCode 632 — Smallest Range Covering Elements from K Lists bằng Python

- Link: https://leetcode.com/problems/smallest-range-covering-elements-from-k-lists/
- Độ khó: **Hard**
- Chủ đề: Heap (min-heap), K con trỏ (k-way merge), Cửa sổ trượt (sliding window), Sắp xếp, Tìm kiếm nhị phân

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def smallestRange(self, nums: List[List[int]]) -> List[int]:

```

## 1. Giới thiệu & mục tiêu học

Bài này là sự kết hợp đẹp của hai kỹ thuật kinh điển: **trộn k danh sách đã sắp xếp bằng heap** (k-way merge) và **cửa sổ trượt nhỏ nhất bao đủ mọi loại** (giống "Minimum Window Substring"). Nếu bạn đã biết Merge k Sorted Lists, bài này là bước tiếp theo rất tự nhiên.

Sau notebook này bạn sẽ:

1. Hiểu chính xác thế nào là "đoạn `[a, b]` phủ mọi danh sách" và luật so sánh hai đoạn (độ dài trước, đầu mút trái sau).
2. Viết lời giải vét cạn mọi cặp `(a, b)` và lời giải "cố định đầu trái, tìm đầu phải nhỏ nhất bằng `bisect`".
3. Nắm kỹ thuật **k con trỏ + min-heap**: luôn đẩy con trỏ của phần tử nhỏ nhất.
4. Nắm kỹ thuật **trộn tất cả rồi cửa sổ trượt** với bộ đếm "số danh sách đã phủ".
5. Tự chứng minh vì sao chỉ cần xét đoạn có đầu trái là phần tử nhỏ nhất hiện tại, và kiểm chứng bằng brute force có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `k` danh sách số nguyên, **mỗi danh sách đã sắp xếp không giảm**. Tìm đoạn `[a, b]` **nhỏ nhất** sao cho **mỗi danh sách có ít nhất một phần tử** nằm trong đoạn (tức là `a <= x <= b`).

Luật so sánh: đoạn `[a, b]` **nhỏ hơn** đoạn `[c, d]` nếu
- `b - a < d - c`, **hoặc**
- `b - a == d - c` **và** `a < c`.

Nói cách khác: so sánh theo cặp `(b - a, a)` theo thứ tự từ điển.

**Ví dụ 1**

```text
Input:  nums = [[4,10,15,24,26],[0,9,12,20],[5,18,22,30]]
Output: [20,24]
Giải thích:
  Danh sách 1: [4, 10, 15, 24, 26] -> 24 nằm trong [20, 24]
  Danh sách 2: [0, 9, 12, 20]      -> 20 nằm trong [20, 24]
  Danh sách 3: [5, 18, 22, 30]     -> 22 nằm trong [20, 24]
```

**Ví dụ 2**

```text
Input:  nums = [[1,2,3],[1,2,3],[1,2,3]]
Output: [1,1]
```

**Ràng buộc**

- `nums.length == k`, `1 <= k <= 3500`
- `1 <= nums[i].length <= 50`
- `-10^5 <= nums[i][j] <= 10^5`
- `nums[i]` sắp xếp không giảm.

Tổng số phần tử `N <= 3500 · 50 = 175 000`. Lời giải `O(N log k)` hoặc `O(N log N)` là thoải mái; `O(N^2)` thì không.

In [1]:
def covers(nums, a, b):
    """True nếu mọi danh sách có ít nhất một phần tử trong [a, b]."""
    return all(any(a <= x <= b for x in lst) for lst in nums)

ex1 = [[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]
ex2 = [[1, 2, 3], [1, 2, 3], [1, 2, 3]]
print("[20,24] phủ ex1?", covers(ex1, 20, 24))
print("[20,23] phủ ex1?", covers(ex1, 20, 23))   # danh sách 1 không có số nào trong [20,23]
print("[0,5]   phủ ex1?", covers(ex1, 0, 5))     # độ dài 5 nhưng thiếu? kiểm tra
print("[1,1]   phủ ex2?", covers(ex2, 1, 1))

[20,24] phủ ex1? True
[20,23] phủ ex1? False
[0,5]   phủ ex1? True
[1,1]   phủ ex2? True


Để ý `[0, 5]`: danh sách 1 có 4, danh sách 2 có 0, danh sách 3 có 5 → **phủ được**, độ dài `5 - 0 = 5`. Nhưng `[20, 24]` có độ dài `4 < 5` nên vẫn tốt hơn. Những phép thử tay như thế này giúp ta tin vào đáp án của đề.

## 3. Kiến thức nền

### 3.1. So sánh tuple trong Python

Python so sánh tuple **từ trái sang phải**: so phần tử đầu; nếu bằng nhau mới so phần tử thứ hai. Vì vậy luật "độ dài trước, đầu trái sau" chỉ là so sánh `(b - a, a)`.

In [2]:
def key_of(rng):
    a, b = rng
    return (b - a, a)

print(key_of([20, 24]), key_of([0, 5]))
print((4, 20) < (5, 0))          # True: độ dài 4 < 5
print((4, 20) < (4, 21))         # True: cùng độ dài, đầu trái 20 < 21
print(min([[0, 5], [20, 24], [10, 14]], key=key_of))   # [10, 14]

(4, 20) (5, 0)
True
True
[10, 14]


### 3.2. `heapq` — min-heap

`heapq` biến một list thành **min-heap**: `h[0]` luôn là phần tử nhỏ nhất. `heappush` và `heappop` tốn `O(log size)`. Khi đẩy **tuple** `(giá trị, chỉ số danh sách, chỉ số phần tử)`, heap so sánh theo giá trị trước — đúng thứ ta cần.

In [3]:
import heapq
h = []
for item in [(10, 0, 1), (4, 1, 0), (7, 2, 3)]:
    heapq.heappush(h, item)
print("nhỏ nhất:", h[0])
print("lấy ra lần lượt:", [heapq.heappop(h) for _ in range(3)])

nhỏ nhất: (4, 1, 0)
lấy ra lần lượt: [(4, 1, 0), (7, 2, 3), (10, 0, 1)]


### 3.3. `heapq.heapify` và `heapq.merge`

- `heapify(list)` biến list sẵn có thành heap trong `O(n)`.
- `heapq.merge(*iterables)` trộn nhiều dãy đã sắp thành một dãy sắp xếp — chính là k-way merge viết sẵn.

In [4]:
h = [(5, 2), (1, 0), (3, 1)]
heapq.heapify(h)
print(h[0])
print(list(heapq.merge([1, 4, 9], [2, 3, 10], [0, 11])))

(1, 0)
[0, 1, 2, 3, 4, 9, 10, 11]


### 3.4. `bisect_left` — tìm phần tử đầu tiên `>= x`

Trong list đã sắp, `bisect_left(lst, x)` trả về vị trí `i` nhỏ nhất mà `lst[i] >= x` (bằng `len(lst)` nếu không có). Độ phức tạp `O(log n)`.

In [5]:
from bisect import bisect_left
lst = [4, 10, 15, 24, 26]
for x in [3, 10, 11, 26, 27]:
    i = bisect_left(lst, x)
    print(f"x={x:>2}: i={i}, phần tử đầu tiên >= x:", lst[i] if i < len(lst) else "không có")

x= 3: i=0, phần tử đầu tiên >= x: 4
x=10: i=1, phần tử đầu tiên >= x: 10
x=11: i=2, phần tử đầu tiên >= x: 15
x=26: i=4, phần tử đầu tiên >= x: 26
x=27: i=5, phần tử đầu tiên >= x: không có


### 3.5. Gắn nhãn danh sách cho phần tử: list các cặp `(giá trị, nhãn)`

Cách 4 sẽ "trộn" tất cả phần tử vào **một** list nhưng vẫn nhớ mỗi phần tử thuộc danh sách nào. Ta tạo cặp `(value, list_id)` rồi sắp xếp.

In [6]:
pairs = sorted((x, i) for i, lst in enumerate(ex1) for x in lst)
print(pairs[:8], "...")
print("tổng số phần tử:", len(pairs))

[(0, 1), (4, 0), (5, 2), (9, 1), (10, 0), (12, 1), (15, 0), (18, 2)] ...
tổng số phần tử: 13


### 3.6. Bộ đếm `count[list_id]` và biến `covered`

Trong cửa sổ trượt, ta giữ `count[i]` = số phần tử của danh sách `i` đang nằm trong cửa sổ, và `covered` = số danh sách có `count[i] > 0`. Khi `count[i]` từ 0 lên 1 thì `covered += 1`; từ 1 về 0 thì `covered -= 1`.

In [7]:
k = 3
count = [0] * k
covered = 0
for lab in [0, 0, 2, 1]:
    count[lab] += 1
    if count[lab] == 1:
        covered += 1
    print(f"thêm nhãn {lab}: count={count}, covered={covered}")

thêm nhãn 0: count=[1, 0, 0], covered=1
thêm nhãn 0: count=[2, 0, 0], covered=1
thêm nhãn 2: count=[2, 0, 1], covered=2
thêm nhãn 1: count=[2, 1, 1], covered=3


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Đầu mút của đoạn tối ưu luôn là phần tử có thật

Nếu `[a, b]` phủ mọi danh sách, ta có thể **co** `a` lên tới phần tử nhỏ nhất nằm trong đoạn, và **co** `b` xuống tới phần tử lớn nhất nằm trong đoạn — đoạn vẫn phủ, mà không dài hơn. Vậy chỉ cần xét `a`, `b` là **giá trị xuất hiện trong input**.

### 4.2. "Chọn một đại diện cho mỗi danh sách"

Một đoạn phủ ⇔ ta chọn được **một phần tử từ mỗi danh sách** (gọi là đại diện), và đoạn chính là `[min đại diện, max đại diện]`. Bài toán trở thành: chọn mỗi danh sách một phần tử sao cho `max - min` nhỏ nhất.

### 4.3. Tính tay với ví dụ nhỏ

`nums = [[1, 5], [3, 8], [4, 9]]`. Các cách chọn đại diện (2·2·2 = 8 cách):

| Đại diện | min | max | độ dài |
|---|---|---|---|
| 1, 3, 4 | 1 | 4 | 3 |
| 1, 3, 9 | 1 | 9 | 8 |
| 1, 8, 4 | 1 | 8 | 7 |
| 1, 8, 9 | 1 | 9 | 8 |
| 5, 3, 4 | 3 | 5 | **2** |
| 5, 3, 9 | 3 | 9 | 6 |
| 5, 8, 4 | 4 | 8 | 4 |
| 5, 8, 9 | 5 | 9 | 4 |

Đáp án `[3, 5]`. Để ý: với đại diện `1, 3, 4`, phần tử **nhỏ nhất là 1**. Muốn đoạn ngắn hơn, nâng `max` lên chỉ làm tệ hơn — cách duy nhất là **bỏ phần tử nhỏ nhất, thay bằng phần tử kế tiếp trong cùng danh sách** (1 → 5). Đây là mầm mống của Cách 3.

In [8]:
from itertools import product

small = [[1, 5], [3, 8], [4, 9]]
rows = []
for reps in product(*small):
    rows.append((max(reps) - min(reps), min(reps), reps))
for width, lo, reps in sorted(rows):
    print(f"đại diện {reps}: đoạn [{lo}, {lo + width}], độ dài {width}")

đại diện (5, 3, 4): đoạn [3, 5], độ dài 2
đại diện (1, 3, 4): đoạn [1, 4], độ dài 3
đại diện (5, 8, 4): đoạn [4, 8], độ dài 4
đại diện (5, 8, 9): đoạn [5, 9], độ dài 4
đại diện (5, 3, 9): đoạn [3, 9], độ dài 6
đại diện (1, 8, 4): đoạn [1, 8], độ dài 7
đại diện (1, 3, 9): đoạn [1, 9], độ dài 8
đại diện (1, 8, 9): đoạn [1, 9], độ dài 8


## 5. Cách 1 — Brute Force: thử mọi cặp `(a, b)`

**Ý tưởng.** Theo 4.1, `a` và `b` là giá trị có trong input. Gom các giá trị khác nhau `V`, thử mọi cặp `a <= b`, kiểm tra `covers`, giữ đoạn có khóa `(b - a, a)` nhỏ nhất.

Để code đơn giản và chắc chắn đúng (dùng làm "trọng tài" đối chiếu), ta không tối ưu gì cả.

In [9]:
def smallest_range_brute(nums):
    vals = sorted(set(x for lst in nums for x in lst))
    best = None
    for a in vals:
        for b in vals:
            if b < a:
                continue
            if best is not None and (b - a, a) >= (best[1] - best[0], best[0]):
                continue                     # không thể tốt hơn, khỏi kiểm tra
            if covers(nums, a, b):
                best = [a, b]
    return best

print(smallest_range_brute(ex1), smallest_range_brute(ex2), smallest_range_brute(small))
assert smallest_range_brute(ex1) == [20, 24]
assert smallest_range_brute(ex2) == [1, 1]
assert smallest_range_brute(small) == [3, 5]

[20, 24] [1, 1] [3, 5]


**Độ phức tạp.** Có `V <= N` giá trị → `O(V^2)` cặp, mỗi lần `covers` tốn `O(N)` → **`O(N^3)`** trong trường hợp xấu. Với `N = 175 000` là bất khả thi. Space `O(N)`.

**Nút thắt.** Với mỗi `a` ta thử **mọi** `b`. Nhưng khi đã cố định `a`, đầu phải tốt nhất là **xác định được ngay**: mỗi danh sách phải có một phần tử `>= a`, ta chọn phần tử nhỏ nhất như vậy; `b` là max của các phần tử đó.

In [10]:
import random, time
random.seed(1)
def rand_nums(k, maxlen, lo, hi):
    return [sorted(random.randint(lo, hi) for _ in range(random.randint(1, maxlen))) for _ in range(k)]

for k in (4, 8, 12):
    nums = rand_nums(k, 6, -30, 30)
    t = time.perf_counter()
    smallest_range_brute(nums)
    print(f"k={k}, N={sum(map(len, nums))}: {time.perf_counter() - t:.4f}s")

k=4, N=8: 0.0000s
k=8, N=24: 0.0002s
k=12, N=46: 0.0007s


## 6. Bước đột phá

**Insight 1 (cố định đầu trái).** Nếu đầu trái là `a`, thì với mỗi danh sách `i`, phần tử "tốt nhất" để làm đại diện là **phần tử nhỏ nhất `>= a`** (gọi là `next_i(a)`). Khi đó đầu phải nhỏ nhất là `b(a) = max_i next_i(a)`. Nếu danh sách nào không có phần tử `>= a` thì không còn đoạn nào bắt đầu từ `a` trở đi.

**Insight 2 (đẩy con trỏ nhỏ nhất).** Giữ một con trỏ cho mỗi danh sách. Tập đại diện hiện tại cho đoạn `[min, max]`. Để có đoạn khác **có thể** tốt hơn, bắt buộc phải bỏ phần tử `min` (giữ `min` thì đoạn luôn chứa `[min, max]` hiện tại hoặc dài hơn). Vậy: **dời con trỏ của danh sách chứa `min` sang phải một bước**. Lặp đến khi một danh sách hết phần tử.

**Invariant của Cách 3:** tại mọi thời điểm, mỗi danh sách có đúng một đại diện, và các đại diện bị bỏ qua (bên trái con trỏ) **không thể** là đầu trái của một đoạn tốt hơn đoạn tốt nhất đã ghi nhận.

**Quy tắc nếu…thì…:**
- Nếu `max - min` hiện tại nhỏ hơn (theo khóa `(độ dài, đầu trái)`) đoạn tốt nhất thì cập nhật.
- Nếu danh sách chứa `min` đã hết phần tử thì **dừng** — không thể bỏ `min` được nữa.
- Nếu không thì thay `min` bằng phần tử kế tiếp trong danh sách đó, cập nhật `max` nếu phần tử mới lớn hơn.

**Insight 3 (nhìn theo cửa sổ).** Trộn tất cả thành một dãy `(giá trị, nhãn)` đã sắp. Đoạn phủ ⇔ một **đoạn con liên tiếp** của dãy chứa đủ `k` nhãn. Đó là bài cửa sổ trượt nhỏ nhất chứa đủ mọi loại.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time |
|---|---|---|
| 1 | Brute force mọi cặp (mục 5) | `O(N^3)` |
| 2 | Cố định đầu trái + `bisect` | `O(V · k · log L)` |
| 3 | K con trỏ + min-heap | `O(N log k)` |
| 4 | Trộn + cửa sổ trượt | `O(N log N)` |

(`V` = số giá trị khác nhau, `L` = độ dài lớn nhất của một danh sách.)

### 7.1. Cách 2 — Cố định đầu trái, tìm đầu phải bằng `bisect`

**Giải thích.** Với mỗi giá trị `a` trong `V` (tăng dần), tính `b(a) = max_i nums[i][bisect_left(nums[i], a)]`. Nếu có danh sách nào `bisect_left` trả về `len` thì dừng (mọi `a` lớn hơn cũng thất bại). Lấy đoạn có khóa nhỏ nhất.

In [11]:
from bisect import bisect_left

def smallest_range_bisect(nums):
    vals = sorted(set(x for lst in nums for x in lst))
    best = None
    for a in vals:
        b = a
        for lst in nums:
            j = bisect_left(lst, a)
            if j == len(lst):           # danh sách này không còn phần tử >= a
                return best
            b = max(b, lst[j])
        if best is None or (b - a, a) < (best[1] - best[0], best[0]):
            best = [a, b]
    return best

print(smallest_range_bisect(ex1), smallest_range_bisect(ex2), smallest_range_bisect(small))
assert smallest_range_bisect(ex1) == [20, 24]
assert smallest_range_bisect(small) == [3, 5]

[20, 24] [1, 1] [3, 5]


**Đọc code:**

- `vals` tăng dần nên khi một danh sách "cạn" (không còn phần tử `>= a`) thì mọi `a` sau cũng cạn → `return best` sớm.
- `b` bắt đầu bằng `a`: vì `a` có trong ít nhất một danh sách, và mọi `lst[j] >= a`, nên max cuối cùng `>= a`.
- So sánh tuple `(b - a, a)` thể hiện đúng luật của đề.

**Code nộp LeetCode (Cách 2):** `V` có thể tới `2·10^5 + 1`, `k = 3500` → `7·10^8` lần `bisect` → **TLE**. Nhưng đây là bước đệm tư duy quan trọng.

In [12]:
from typing import List

class SolutionBisect:
    def smallestRange(self, nums: List[List[int]]) -> List[int]:
        vals = sorted({x for lst in nums for x in lst})
        best = [vals[0], vals[-1]]
        for a in vals:
            b = a
            for lst in nums:
                j = bisect_left(lst, a)
                if j == len(lst):
                    return best
                if lst[j] > b:
                    b = lst[j]
            if b - a < best[1] - best[0]:      # a tăng dần nên "<" giữ đầu trái nhỏ nhất
                best = [a, b]
        return best

assert SolutionBisect().smallestRange(ex1) == [20, 24]
assert SolutionBisect().smallestRange(ex2) == [1, 1]
print("SolutionBisect OK")

SolutionBisect OK


Chú ý chi tiết tinh tế: khởi tạo `best = [vals[0], vals[-1]]` là hợp lệ vì đoạn từ giá trị nhỏ nhất tới lớn nhất chắc chắn phủ mọi danh sách. Vì `a` duyệt tăng dần, so sánh **nghiêm ngặt** `<` theo độ dài đủ để giữ đầu trái nhỏ nhất khi hòa.

**Độ phức tạp Cách 2:** Time `O(N log N + V · k · log L)`, Space `O(V)`.

### 7.2. Cách 3 — K con trỏ + min-heap (lời giải chuẩn)

**Giải thích.** Cài đặt Insight 2:

```text
heap = [(nums[i][0], i, 0) với mọi i];  cur_max = max(nums[i][0])
best = [-inf, +inf]
lặp:
    (lo, i, j) = heap[0]                 # phần tử nhỏ nhất
    nếu cur_max - lo < độ dài best: best = [lo, cur_max]
    nếu j + 1 == len(nums[i]): dừng
    pop; push (nums[i][j+1], i, j+1); cur_max = max(cur_max, nums[i][j+1])
```

Heap giúp tìm `min` của `k` đại diện trong `O(log k)`; `max` thì chỉ tăng nên giữ bằng một biến.

In [13]:
import heapq

def smallest_range_heap(nums):
    heap = [(lst[0], i, 0) for i, lst in enumerate(nums)]
    heapq.heapify(heap)
    cur_max = max(lst[0] for lst in nums)
    best = [heap[0][0], cur_max]
    while True:
        lo, i, j = heap[0]
        if cur_max - lo < best[1] - best[0]:
            best = [lo, cur_max]
        if j + 1 == len(nums[i]):
            return best
        nxt = nums[i][j + 1]
        heapq.heapreplace(heap, (nxt, i, j + 1))   # pop rồi push trong một bước
        if nxt > cur_max:
            cur_max = nxt

print(smallest_range_heap(ex1), smallest_range_heap(ex2), smallest_range_heap(small))
assert smallest_range_heap(ex1) == [20, 24]
assert smallest_range_heap(ex2) == [1, 1]
assert smallest_range_heap(small) == [3, 5]

[20, 24] [1, 1] [3, 5]


**Đọc code từng dòng:**

- `heap = [(lst[0], i, 0) ...]`: mỗi danh sách góp phần tử đầu tiên; bộ ba lưu giá trị, danh sách nào, vị trí nào.
- `heapify`: `O(k)`.
- `best = [heap[0][0], cur_max]`: đoạn đầu tiên đã phủ (mỗi danh sách có đại diện).
- `cur_max - lo < best[1] - best[0]`: chỉ cập nhật khi **ngắn hơn hẳn**. Vì `lo` (min hiện tại) **không giảm** theo thời gian, nếu hòa độ dài thì đoạn cũ có đầu trái nhỏ hơn hoặc bằng → giữ đoạn cũ là đúng luật.
- `heapreplace`: lấy đỉnh ra và đẩy phần tử mới vào, nhanh hơn `heappop` + `heappush`.
- Dừng khi danh sách chứa `min` hết phần tử.

**Tại sao `lo` không giảm?** Phần tử mới đẩy vào luôn `>=` phần tử vừa bỏ (cùng danh sách, đã sắp), nên min của tập đại diện không thể giảm.

**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionHeap:
    def smallestRange(self, nums: List[List[int]]) -> List[int]:
        heap = [(row[0], i, 0) for i, row in enumerate(nums)]
        heapq.heapify(heap)
        cur_max = max(row[0] for row in nums)
        best_lo, best_hi = heap[0][0], cur_max
        while True:
            lo, i, j = heap[0]
            if cur_max - lo < best_hi - best_lo:
                best_lo, best_hi = lo, cur_max
            if j + 1 == len(nums[i]):
                return [best_lo, best_hi]
            nxt = nums[i][j + 1]
            heapq.heapreplace(heap, (nxt, i, j + 1))
            if nxt > cur_max:
                cur_max = nxt

assert SolutionHeap().smallestRange(ex1) == [20, 24]
assert SolutionHeap().smallestRange(ex2) == [1, 1]
print("SolutionHeap OK")

SolutionHeap OK


**Độ phức tạp Cách 3:** mỗi phần tử vào heap tối đa một lần, mỗi thao tác `O(log k)` → Time **`O(N log k)`**; Space **`O(k)`** cho heap.

### 7.3. Cách 4 — Trộn tất cả + cửa sổ trượt

**Giải thích.** Tạo `pairs` = mọi `(giá trị, nhãn danh sách)` đã sắp theo giá trị. Dùng hai con trỏ `left`, `right`:

```text
với right chạy từ 0:
    thêm pairs[right] vào cửa sổ (count[nhãn] += 1, có thể covered += 1)
    trong khi covered == k:
        đoạn [pairs[left].giá trị, pairs[right].giá trị] là ứng viên
        bỏ pairs[left] khỏi cửa sổ, left += 1
```

Cửa sổ đủ `k` nhãn ⇔ đoạn giá trị tương ứng phủ mọi danh sách. Thu hẹp bên trái tối đa để có đoạn ngắn nhất với `right` cố định.

In [15]:
def smallest_range_window(nums):
    k = len(nums)
    pairs = sorted((x, i) for i, lst in enumerate(nums) for x in lst)
    count = [0] * k
    covered = 0
    best = None
    left = 0
    for right, (x, lab) in enumerate(pairs):
        count[lab] += 1
        if count[lab] == 1:
            covered += 1
        while covered == k:
            a = pairs[left][0]
            if best is None or x - a < best[1] - best[0]:
                best = [a, x]
            lab_l = pairs[left][1]
            count[lab_l] -= 1
            if count[lab_l] == 0:
                covered -= 1
            left += 1
    return best

print(smallest_range_window(ex1), smallest_range_window(ex2), smallest_range_window(small))
assert smallest_range_window(ex1) == [20, 24]
assert smallest_range_window(ex2) == [1, 1]

[20, 24] [1, 1] [3, 5]


**Đọc code:**

- `sorted((x, i) ...)`: khi giá trị bằng nhau, nhãn nhỏ đứng trước — không ảnh hưởng đúng sai.
- Vòng `while covered == k`: mỗi lần lặp ghi nhận ứng viên rồi bỏ phần tử trái. `left` chỉ tăng, nên tổng số lần lặp `while` là `O(N)`.
- So sánh `x - a < ...` nghiêm ngặt: các ứng viên xuất hiện với `right` tăng dần; nếu hai đoạn cùng độ dài, đoạn đến trước có `b` nhỏ hơn hoặc bằng nên `a` nhỏ hơn hoặc bằng → giữ đoạn cũ là đúng.

**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionWindow:
    def smallestRange(self, nums: List[List[int]]) -> List[int]:
        k = len(nums)
        pairs = sorted((x, i) for i, row in enumerate(nums) for x in row)
        count = [0] * k
        covered = 0
        left = 0
        best_lo, best_hi = pairs[0][0], pairs[-1][0]
        for x, lab in pairs:
            count[lab] += 1
            if count[lab] == 1:
                covered += 1
            while covered == k:
                a, lab_l = pairs[left]
                if x - a < best_hi - best_lo:
                    best_lo, best_hi = a, x
                count[lab_l] -= 1
                if count[lab_l] == 0:
                    covered -= 1
                left += 1
        return [best_lo, best_hi]

assert SolutionWindow().smallestRange(ex1) == [20, 24]
assert SolutionWindow().smallestRange(ex2) == [1, 1]
print("SolutionWindow OK")

SolutionWindow OK


**Độ phức tạp Cách 4:** sắp xếp `O(N log N)`, cửa sổ `O(N)` → Time **`O(N log N)`**; Space **`O(N)`** cho `pairs`.

Lưu ý về khởi tạo `best = [min toàn cục, max toàn cục]`: đoạn này luôn phủ, nên vòng `while` sẽ chỉ cập nhật khi tìm được đoạn ngắn hơn hẳn. Nếu đoạn toàn cục chính là đáp án (ví dụ `k = 1` với một phần tử), ta trả đúng nó.

## 8. TRACE từng bước (Cách 3 — heap)

Trace ví dụ 1. Ký hiệu đại diện của 3 danh sách và đoạn `[min, max]`:

| Bước | Đại diện (L1, L2, L3) | min (từ DS) | max | Độ dài | best |
|---|---|---|---|---|---|
| 0 | 4, 0, 5 | 0 (L2) | 5 | 5 | [0,5] |
| 1 | 4, 9, 5 | 4 (L1) | 9 | 5 | [0,5] |
| 2 | 10, 9, 5 | 5 (L3) | 10 | 5 | [0,5] |
| 3 | 10, 9, 18 | 9 (L2) | 18 | 9 | [0,5] |
| 4 | 10, 12, 18 | 10 (L1) | 18 | 8 | [0,5] |
| 5 | 15, 12, 18 | 12 (L2) | 18 | 6 | [0,5] |
| 6 | 15, 20, 18 | 15 (L1) | 20 | 5 | [0,5] |
| 7 | 24, 20, 18 | 18 (L3) | 24 | 6 | [0,5] |
| 8 | 24, 20, 22 | 20 (L2) | 24 | **4** | **[20,24]** |
| 9 | L2 hết phần tử → dừng | | | | [20,24] |

Ở bước 6, độ dài 5 bằng best nhưng đầu trái 15 > 0 nên **không** cập nhật — đúng luật hòa.

In [17]:
def trace_heap(nums):
    heap = [(lst[0], i, 0) for i, lst in enumerate(nums)]
    heapq.heapify(heap)
    reps = [lst[0] for lst in nums]
    cur_max = max(reps)
    best = [heap[0][0], cur_max]
    step = 0
    print(f"{'bước':>4} | {'đại diện':<18} | {'min':>4} {'DS':>3} | {'max':>4} | {'dài':>3} | best")
    while True:
        lo, i, j = heap[0]
        if cur_max - lo < best[1] - best[0]:
            best = [lo, cur_max]
        print(f"{step:>4} | {str(reps):<18} | {lo:>4} L{i + 1:<2}| {cur_max:>4} | {cur_max - lo:>3} | {best}")
        if j + 1 == len(nums[i]):
            print(f"  -> danh sách L{i + 1} hết phần tử, dừng.")
            return best
        nxt = nums[i][j + 1]
        heapq.heapreplace(heap, (nxt, i, j + 1))
        reps[i] = nxt
        cur_max = max(cur_max, nxt)
        step += 1

print("Kết quả:", trace_heap(ex1))

bước | đại diện           |  min  DS |  max | dài | best
   0 | [4, 0, 5]          |    0 L2 |    5 |   5 | [0, 5]
   1 | [4, 9, 5]          |    4 L1 |    9 |   5 | [0, 5]
   2 | [10, 9, 5]         |    5 L3 |   10 |   5 | [0, 5]
   3 | [10, 9, 18]        |    9 L2 |   18 |   9 | [0, 5]
   4 | [10, 12, 18]       |   10 L1 |   18 |   8 | [0, 5]
   5 | [15, 12, 18]       |   12 L2 |   18 |   6 | [0, 5]
   6 | [15, 20, 18]       |   15 L1 |   20 |   5 | [0, 5]
   7 | [24, 20, 18]       |   18 L3 |   24 |   6 | [0, 5]
   8 | [24, 20, 22]       |   20 L2 |   24 |   4 | [20, 24]
  -> danh sách L2 hết phần tử, dừng.
Kết quả: [20, 24]


Trace thêm Cách 4 (cửa sổ) cho ví dụ nhỏ `[[1,5],[3,8],[4,9]]` để thấy `left`, `right`, `covered` thay đổi:

In [18]:
def trace_window(nums):
    k = len(nums)
    pairs = sorted((x, i) for i, lst in enumerate(nums) for x in lst)
    print("pairs =", pairs)
    count = [0] * k
    covered, left, best = 0, 0, None
    for right, (x, lab) in enumerate(pairs):
        count[lab] += 1
        covered += count[lab] == 1
        print(f"right={right} thêm {(x, lab)} -> count={count}, covered={covered}")
        while covered == k:
            a = pairs[left][0]
            if best is None or x - a < best[1] - best[0]:
                best = [a, x]
            print(f"    đủ {k} nhãn: ứng viên [{a}, {x}] (dài {x - a}), best={best}; bỏ {pairs[left]}")
            lab_l = pairs[left][1]
            count[lab_l] -= 1
            covered -= count[lab_l] == 0
            left += 1
    return best

print("Kết quả:", trace_window(small))

pairs = [(1, 0), (3, 1), (4, 2), (5, 0), (8, 1), (9, 2)]
right=0 thêm (1, 0) -> count=[1, 0, 0], covered=1
right=1 thêm (3, 1) -> count=[1, 1, 0], covered=2
right=2 thêm (4, 2) -> count=[1, 1, 1], covered=3
    đủ 3 nhãn: ứng viên [1, 4] (dài 3), best=[1, 4]; bỏ (1, 0)
right=3 thêm (5, 0) -> count=[1, 1, 1], covered=3
    đủ 3 nhãn: ứng viên [3, 5] (dài 2), best=[3, 5]; bỏ (3, 1)
right=4 thêm (8, 1) -> count=[1, 1, 1], covered=3
    đủ 3 nhãn: ứng viên [4, 8] (dài 4), best=[3, 5]; bỏ (4, 2)
right=5 thêm (9, 2) -> count=[1, 1, 1], covered=3
    đủ 3 nhãn: ứng viên [5, 9] (dài 4), best=[3, 5]; bỏ (5, 0)
Kết quả: [3, 5]


## 9. Code nộp LeetCode

Lời giải chính: **K con trỏ + min-heap** (Cách 3), `O(N log k)` thời gian, `O(k)` bộ nhớ.

In [19]:
from typing import List
import heapq

class Solution:
    def smallestRange(self, nums: List[List[int]]) -> List[int]:
        # Mỗi danh sách góp một đại diện: (giá trị, chỉ số danh sách, chỉ số phần tử)
        heap = [(row[0], i, 0) for i, row in enumerate(nums)]
        heapq.heapify(heap)
        cur_max = max(row[0] for row in nums)
        best_lo, best_hi = heap[0][0], cur_max

        while True:
            lo, i, j = heap[0]                       # đại diện nhỏ nhất
            if cur_max - lo < best_hi - best_lo:     # ngắn hơn hẳn mới cập nhật
                best_lo, best_hi = lo, cur_max
            if j + 1 == len(nums[i]):                # không thể bỏ min nữa
                return [best_lo, best_hi]
            nxt = nums[i][j + 1]
            heapq.heapreplace(heap, (nxt, i, j + 1)) # thay min bằng phần tử kế tiếp
            if nxt > cur_max:
                cur_max = nxt

sol = Solution()
assert sol.smallestRange(ex1) == [20, 24]
assert sol.smallestRange(ex2) == [1, 1]
assert sol.smallestRange([[1]]) == [1, 1]
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Mệnh đề.** Gọi `[a*, b*]` là đáp án tối ưu (độ dài nhỏ nhất, hòa thì đầu trái nhỏ nhất). Cách 3 trả về đúng `[a*, b*]`.

**Chứng minh.**

1. **Mọi đoạn được ghi nhận đều phủ** (mỗi danh sách có đại diện trong `[min, max]`), nên kết quả không thể tốt hơn tối ưu.
2. Với mỗi danh sách `t`, gọi `p_t` là **vị trí đầu tiên** có `nums[t][p_t] >= a*`. Vì `[a*, b*]` phủ danh sách `t`, phần tử đó tồn tại và `<= b*`. Danh sách chứa giá trị `a*` có `nums[t][p_t] = a*`.
3. **Bổ đề:** chừng nào còn con trỏ nào đứng **trước** vị trí `p_t` của nó, thuật toán không dừng và không con trỏ nào vượt quá `p_t`.
   - Con trỏ đứng trước `p_t` trỏ vào phần tử `< a*`. Con trỏ đứng đúng tại `p_t` (hoặc sau) trỏ vào phần tử `>= a*`. Vậy **min** của tập đại diện là phần tử `< a*`, thuộc một danh sách có con trỏ còn đứng trước đích.
   - Danh sách đó còn phần tử kế tiếp (ít nhất là phần tử ở `p_t`), nên thuật toán không dừng; con trỏ tiến một bước nên không vượt `p_t`.
4. Mỗi bước, tổng vị trí con trỏ tăng 1, nên sau hữu hạn bước mọi con trỏ đứng **đúng** tại `p_t`. Lúc đó mọi đại diện nằm trong `[a*, b*]` và có một đại diện bằng `a*` → đoạn hiện tại là `[a*, m]` với `m <= b*`; do tối ưu, `m = b*`. Đoạn này được xét để cập nhật.
5. **Luật hòa:** `min` không giảm theo thời gian. Đoạn ghi nhận trước thời điểm ở bước 4 có đầu trái `<= a*`; nếu nó cùng độ dài tối ưu thì đầu trái không thể `< a*` (mâu thuẫn tối ưu) nên chính là `[a*, b*]`. Sau đó chỉ cập nhật khi **ngắn hơn hẳn** — không thể xảy ra. Vậy kết quả là `[a*, b*]`.

Tóm tắt trực giác: **bỏ phần tử nhỏ nhất là nước đi duy nhất có thể cải thiện**; giữ nó lại thì mọi đoạn sau đều chứa nó và chứa max hiện tại.

**Cách 4 đúng vì:** đoạn phủ ⇔ đoạn con liên tiếp của `pairs` chứa đủ `k` nhãn; với mỗi `right`, cửa sổ thu hẹp tối đa cho đoạn ngắn nhất kết thúc tại `pairs[right]` — chuẩn sliding window.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(V^2 · N)` | `O(V)` | chỉ để đối chiếu |
| 2. Bisect | `O(V · k log L)` | `O(V)` | trực quan nhưng TLE |
| 3. Heap | `O(N log k)` | `O(k)` | tối ưu, ít bộ nhớ |
| 4. Window | `O(N log N)` | `O(N)` | dễ nhớ nếu đã biết sliding window |

Với `N = 175 000`, `k = 3500`: Cách 3 cỡ `175 000 · 12 ≈ 2·10^6` phép heap — rất nhanh. Cách 4 sắp xếp 175 000 cặp cũng chỉ mất vài chục ms.

In [20]:
random.seed(7)
big = [sorted(random.randint(-10**5, 10**5) for _ in range(50)) for _ in range(3500)]
for name, f in [("heap", smallest_range_heap), ("window", smallest_range_window)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:>6}: {r}  {time.perf_counter() - t:.3f}s")
assert smallest_range_heap(big) == smallest_range_window(big)

  heap: [60207, 84225]  0.097s


window: [60207, 84225]  0.161s


## 12. Bộ test

Gồm: ví dụ đề, edge case (một danh sách, một phần tử, giá trị âm, trùng lặp, hòa độ dài) và đối chiếu ngẫu nhiên có seed giữa 4 cách.

In [21]:
def all_methods(nums):
    return {
        "brute": smallest_range_brute(nums),
        "bisect": smallest_range_bisect(nums),
        "heap": smallest_range_heap(nums),
        "window": smallest_range_window(nums),
        "SolutionHeap": SolutionHeap().smallestRange(nums),
        "SolutionWindow": SolutionWindow().smallestRange(nums),
        "SolutionBisect": SolutionBisect().smallestRange(nums),
        "Solution": Solution().smallestRange(nums),
    }

cases = [
    (ex1, [20, 24]),
    (ex2, [1, 1]),
    ([[1]], [1, 1]),
    ([[1, 2, 3]], [1, 1]),                    # một danh sách: đoạn độ dài 0 nhỏ nhất bên trái
    ([[-5, -1], [-3, 2]], [-5, -3]),          # số âm: [-5,-3] và [-3,-1] cùng dài 2 -> đầu trái nhỏ
    ([[1, 10], [5, 14]], [1, 5]),             # hòa độ dài: [1,5] và [10,14] cùng dài 4 -> chọn đầu trái nhỏ
    ([[10, 10], [11, 11]], [10, 11]),         # trùng lặp
    ([[1], [100000]], [1, 100000]),
    ([[1, 5], [3, 8], [4, 9]], [3, 5]),
]
for nums, want in cases:
    res = all_methods(nums)
    for name, got in res.items():
        assert got == want, (nums, name, got, want)
print("Tất cả", len(cases), "test cố định OK")

Tất cả 9 test cố định OK


In [22]:
random.seed(2024)
for trial in range(400):
    k = random.randint(1, 5)
    nums = rand_nums(k, 5, -15, 15)
    res = all_methods(nums)
    want = res["brute"]
    for name, got in res.items():
        assert got == want, (nums, name, got, want)
print("400 test ngẫu nhiên: mọi cách khớp brute force")

400 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` khi cập nhật trong heap → sai luật hòa

**Code sai:** `if cur_max - lo <= best_hi - best_lo:` — khi hòa độ dài, đoạn **sau** (đầu trái lớn hơn) ghi đè đoạn trước.

**Vì sao sai:** đề yêu cầu cùng độ dài thì lấy đầu trái nhỏ hơn. `lo` không giảm, nên đoạn sau có đầu trái `>=`.

In [23]:
def heap_wrong_tie(nums):
    heap = [(lst[0], i, 0) for i, lst in enumerate(nums)]
    heapq.heapify(heap)
    cur_max = max(lst[0] for lst in nums)
    best = [heap[0][0], cur_max]
    while True:
        lo, i, j = heap[0]
        if cur_max - lo <= best[1] - best[0]:      # SAI: <=
            best = [lo, cur_max]
        if j + 1 == len(nums[i]):
            return best
        heapq.heapreplace(heap, (nums[i][j + 1], i, j + 1))
        cur_max = max(cur_max, nums[i][j + 1])

print("sai :", heap_wrong_tie([[1, 10], [5, 14]]))
print("đúng:", smallest_range_heap([[1, 10], [5, 14]]))

sai : [10, 14]
đúng: [1, 5]


**Code đúng:** dùng `<` nghiêm ngặt (như `smallest_range_heap`).

### Lỗi 2: Đẩy con trỏ của danh sách chứa **max** thay vì **min**

**Code sai:** dời con trỏ của phần tử lớn nhất "cho đoạn hẹp lại".

**Vì sao sai:** dời phần tử lớn nhất sang phải chỉ làm `max` tăng — đoạn chỉ dài thêm. Phần tử muốn loại bỏ để thu hẹp là **min**.

In [24]:
def move_max_wrong(nums):
    ptr = [0] * len(nums)
    best = None
    while True:
        reps = [nums[i][ptr[i]] for i in range(len(nums))]
        lo, hi = min(reps), max(reps)
        if best is None or hi - lo < best[1] - best[0]:
            best = [lo, hi]
        i = reps.index(hi)                    # SAI: dời danh sách chứa max
        if ptr[i] + 1 == len(nums[i]):
            return best
        ptr[i] += 1

print("sai :", move_max_wrong(ex1))
print("đúng:", smallest_range_heap(ex1))

sai : [0, 5]
đúng: [20, 24]


**Code đúng:** `i = reps.index(lo)` — dời danh sách chứa **min** (đây chính là Cách 3 không dùng heap, `O(N·k)`).

### Lỗi 3: Cửa sổ trượt đếm phần tử thay vì đếm **danh sách**

**Code sai:** `covered = right - left + 1` hoặc kiểm tra "cửa sổ có ít nhất `k` phần tử".

**Vì sao sai:** cửa sổ có thể chứa nhiều phần tử của cùng một danh sách. Điều kiện phủ là **số nhãn khác nhau** bằng `k`.

In [25]:
def window_wrong(nums):
    k = len(nums)
    pairs = sorted((x, i) for i, lst in enumerate(nums) for x in lst)
    best = None
    left = 0
    for right in range(len(pairs)):
        while right - left + 1 >= k:          # SAI: đếm phần tử
            a, b = pairs[left][0], pairs[right][0]
            if best is None or b - a < best[1] - best[0]:
                best = [a, b]
            left += 1
    return best

print("sai :", window_wrong(ex1), "| phủ thật?", covers(ex1, *window_wrong(ex1)))
print("đúng:", smallest_range_window(ex1))

sai : [9, 12] | phủ thật? False
đúng: [20, 24]


**Code đúng:** dùng `count[nhãn]` và `covered` như Cách 4.

### Lỗi 4: Dừng khi **bất kỳ** danh sách hết, thay vì danh sách chứa min

Một số bạn dừng ngay khi con trỏ nào đó tới phần tử cuối. Sai: danh sách ở phần tử cuối nhưng không chứa min vẫn giữ nguyên được, các danh sách khác vẫn có thể tiến. Chỉ dừng khi **danh sách chứa min** không còn phần tử kế tiếp.

## 14. Mẹo & nhận dạng pattern

- **"Chọn một phần tử từ mỗi nhóm, tối thiểu max − min"** → k con trỏ + heap (đẩy min) hoặc trộn + cửa sổ trượt đủ k loại.
- **"Nhiều danh sách đã sắp"** → nghĩ tới `heapq` với bộ ba `(giá trị, danh sách, vị trí)` như Merge k Sorted Lists.
- **"Đoạn liên tiếp ngắn nhất chứa đủ mọi loại"** → sliding window với `count` và `covered` (Minimum Window Substring).
- `max` của đại diện **chỉ tăng** khi ta chỉ đẩy con trỏ tiến → giữ bằng biến, không cần max-heap.
- Luật hòa "đầu trái nhỏ hơn" thường được thỏa **tự động** bằng cách duyệt tăng dần và so sánh nghiêm ngặt.

## 15. Biến thể

1. **Trả về cả các đại diện** (mỗi danh sách chọn phần tử nào) — lưu bản sao tập đại diện khi cập nhật best.
2. **Chỉ cần phủ ít nhất `m` trong `k` danh sách** — Cách 4 tổng quát hóa trực tiếp: điều kiện `covered >= m`.
3. **Danh sách không sắp xếp** — sắp từng danh sách trước, hoặc dùng Cách 4 (vốn sắp toàn bộ).
4. **Minimum Window Substring (LC 76)** — cùng khung cửa sổ trượt.
5. **Minimize max − min khi chọn từ 3 mảng (LC "Find minimum difference among three arrays")** — trường hợp `k = 3`.

In [26]:
def smallest_range_at_least_m(nums, m):
    """Biến thể 2: đoạn nhỏ nhất phủ ít nhất m danh sách."""
    k = len(nums)
    pairs = sorted((x, i) for i, lst in enumerate(nums) for x in lst)
    count = [0] * k
    covered, left, best = 0, 0, None
    for x, lab in pairs:
        count[lab] += 1
        covered += count[lab] == 1
        while covered >= m:
            a, lab_l = pairs[left]
            if best is None or x - a < best[1] - best[0]:
                best = [a, x]
            count[lab_l] -= 1
            covered -= count[lab_l] == 0
            left += 1
    return best

print(smallest_range_at_least_m(ex1, 3), smallest_range_at_least_m(ex1, 2), smallest_range_at_least_m(ex1, 1))
assert smallest_range_at_least_m(ex1, 3) == [20, 24]

[20, 24] [4, 5] [0, 0]


In [27]:
def smallest_range_with_reps(nums):
    """Biến thể 1: trả về đoạn và đại diện của từng danh sách."""
    heap = [(lst[0], i, 0) for i, lst in enumerate(nums)]
    heapq.heapify(heap)
    reps = [lst[0] for lst in nums]
    cur_max = max(reps)
    best = ([heap[0][0], cur_max], reps[:])
    while True:
        lo, i, j = heap[0]
        if cur_max - lo < best[0][1] - best[0][0]:
            best = ([lo, cur_max], reps[:])
        if j + 1 == len(nums[i]):
            return best
        nxt = nums[i][j + 1]
        heapq.heapreplace(heap, (nxt, i, j + 1))
        reps[i] = nxt
        cur_max = max(cur_max, nxt)

print(smallest_range_with_reps(ex1))

([20, 24], [24, 20, 22])


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi cặp giá trị `(a, b)` | `O(V^2 · N)` | `O(V)` | trọng tài kiểm thử |
| 2. Bisect | cố định `a`, `b = max next_i(a)` | `O(V · k log L)` | `O(V)` | khi `k` nhỏ, để hiểu Insight 1 |
| 3. Heap | k con trỏ, luôn đẩy min | `O(N log k)` | `O(k)` | lời giải chuẩn, ít bộ nhớ |
| 4. Window | trộn + cửa sổ đủ `k` nhãn | `O(N log N)` | `O(N)` | khi quen sliding window; dễ mở rộng "ít nhất m" |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `smallestRange([[1,3,5],[2,4,6]])` và `smallestRange([[10],[1,2,3,20]])`?

**Bài 2 (tính tay).** Lập bảng trace Cách 3 (như mục 8) cho `[[1,4,7],[2,5,8],[3,6,9]]`.

**Bài 3 (điền TODO).** Hoàn thiện `range_key(rng)` trả về khóa so sánh của đoạn, và `better(r1, r2)` trả `True` nếu `r1` nhỏ hơn `r2` theo luật đề.

**Bài 4 (tự viết lại).** Viết lại Cách 3 (`my_smallest_range`) mà không nhìn code.

**Bài 5 (mini challenge).** Viết `min_diff_three(A, B, C)` trả về giá trị nhỏ nhất của `max(a,b,c) - min(a,b,c)` với `a ∈ A, b ∈ B, c ∈ C` (các mảng **chưa sắp**), dùng ý tưởng đẩy con trỏ nhỏ nhất.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def range_key(rng):
    # TODO: trả về tuple (độ dài, đầu trái)
    pass

def better(r1, r2):
    # TODO: so sánh range_key
    pass

# Bài 4 — TODO
def my_smallest_range(nums):
    # TODO: heap (giá trị, i, j); cur_max; dời con trỏ của min; dừng khi danh sách chứa min hết
    pass

# Bài 5 — TODO
def min_diff_three(A, B, C):
    # TODO: sắp A, B, C; ba con trỏ; mỗi bước tăng con trỏ của phần tử nhỏ nhất
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: mỗi danh sách cần một đại diện; tìm cặp hai số gần nhau nhất.
- Mức 2: `[[1,3,5],[2,4,6]]`: các cặp liền nhau hiệu 1: (1,2), (3,2)... đầu trái nhỏ nhất.
- Mức 3: đáp án `[1,2]`; ví dụ thứ hai: `10` bắt buộc, gần nhất trong danh sách 2 là `3` (hiệu 7) hoặc `20` (hiệu 10) → `[3,10]`.

**Bài 2.**
- Mức 1: bắt đầu đại diện `1, 2, 3`, đoạn `[1,3]` dài 2.
- Mức 2: mỗi bước dời min: 1→4, 2→5, 3→6, ...
- Mức 3: mọi đoạn đều dài 2; best giữ `[1,3]`; dừng khi danh sách chứa min (7) hết phần tử.

**Bài 3.**
- Mức 1: `a, b = rng`.
- Mức 2: `return (b - a, a)`.
- Mức 3: `better = range_key(r1) < range_key(r2)`.

**Bài 4.**
- Mức 1: `heapify` các phần tử đầu.
- Mức 2: vòng `while True`, đọc `heap[0]`.
- Mức 3: `heapreplace` và cập nhật `cur_max`.

**Bài 5.**
- Mức 1: sắp ba mảng, `i = j = l = 0`.
- Mức 2: tính hiệu `max - min` của ba phần tử, cập nhật đáp án.
- Mức 3: tăng con trỏ của phần tử nhỏ nhất; dừng khi một mảng hết.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().smallestRange([[1, 3, 5], [2, 4, 6]]) == [1, 2]
assert Solution().smallestRange([[10], [1, 2, 3, 20]]) == [3, 10]

# Bài 2
assert Solution().smallestRange([[1, 4, 7], [2, 5, 8], [3, 6, 9]]) == [1, 3]
trace_heap([[1, 4, 7], [2, 5, 8], [3, 6, 9]])

# Bài 3
def range_key_ans(rng):
    a, b = rng
    return (b - a, a)

def better_ans(r1, r2):
    return range_key_ans(r1) < range_key_ans(r2)

assert better_ans([20, 24], [0, 5])
assert better_ans([1, 5], [10, 14])
assert not better_ans([10, 14], [1, 5])
print("Bài 1–3 OK")

bước | đại diện           |  min  DS |  max | dài | best
   0 | [1, 2, 3]          |    1 L1 |    3 |   2 | [1, 3]
   1 | [4, 2, 3]          |    2 L2 |    4 |   2 | [1, 3]
   2 | [4, 5, 3]          |    3 L3 |    5 |   2 | [1, 3]
   3 | [4, 5, 6]          |    4 L1 |    6 |   2 | [1, 3]
   4 | [7, 5, 6]          |    5 L2 |    7 |   2 | [1, 3]
   5 | [7, 8, 6]          |    6 L3 |    8 |   2 | [1, 3]
   6 | [7, 8, 9]          |    7 L1 |    9 |   2 | [1, 3]
  -> danh sách L1 hết phần tử, dừng.
Bài 1–3 OK


In [30]:
# Bài 4
def my_smallest_range_ans(nums):
    heap = [(row[0], i, 0) for i, row in enumerate(nums)]
    heapq.heapify(heap)
    hi = max(row[0] for row in nums)
    best = [heap[0][0], hi]
    while True:
        lo, i, j = heap[0]
        if hi - lo < best[1] - best[0]:
            best = [lo, hi]
        if j + 1 == len(nums[i]):
            return best
        heapq.heapreplace(heap, (nums[i][j + 1], i, j + 1))
        hi = max(hi, nums[i][j + 1])

# Bài 5
def min_diff_three_ans(A, B, C):
    arrs = [sorted(A), sorted(B), sorted(C)]
    ptr = [0, 0, 0]
    best = float('inf')
    while True:
        vals = [arrs[t][ptr[t]] for t in range(3)]
        best = min(best, max(vals) - min(vals))
        t = vals.index(min(vals))
        ptr[t] += 1
        if ptr[t] == len(arrs[t]):
            return best

random.seed(5)
for _ in range(200):
    nums = rand_nums(random.randint(1, 4), 5, -10, 10)
    assert my_smallest_range_ans(nums) == smallest_range_brute(nums)
    A, B, C = ([random.randint(-20, 20) for _ in range(random.randint(1, 5))] for _ in range(3))
    want = min(max(a, b, c) - min(a, b, c) for a in A for b in B for c in C)
    assert min_diff_three_ans(A, B, C) == want
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template k con trỏ + heap (đẩy min):**

```python
heap = [(row[0], i, 0) for i, row in enumerate(nums)]
heapq.heapify(heap)
hi = max(row[0] for row in nums)
best = [heap[0][0], hi]
while True:
    lo, i, j = heap[0]
    if hi - lo < best[1] - best[0]:
        best = [lo, hi]
    if j + 1 == len(nums[i]):
        return best
    heapq.heapreplace(heap, (nums[i][j + 1], i, j + 1))
    hi = max(hi, nums[i][j + 1])
```

**Template cửa sổ đủ k loại:**

```python
pairs = sorted((x, i) for i, row in enumerate(nums) for x in row)
count = [0]*k; covered = left = 0
for x, lab in pairs:
    count[lab] += 1; covered += count[lab] == 1
    while covered == k:
        ... ghi nhận [pairs[left][0], x] ...
        count[pairs[left][1]] -= 1; covered -= count[pairs[left][1]] == 0; left += 1
```

**Checklist:**

- [ ] Đẩy con trỏ của **min**, không phải max.
- [ ] Dừng khi danh sách **chứa min** hết phần tử.
- [ ] So sánh độ dài **nghiêm ngặt** `<` để giữ đầu trái nhỏ nhất khi hòa.
- [ ] Bộ ba trong heap có chỉ số danh sách + vị trí (không chỉ giá trị).
- [ ] Cửa sổ trượt đếm **số danh sách** đã phủ, không đếm số phần tử.
- [ ] Kiểm tra `k = 1`, danh sách một phần tử, số âm, phần tử trùng.